Import libraries

In [1]:
import warnings

import cmle
import random
import numpy as np
import pandas as pd
import re
from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Set random seeds for reproducible results across all runs
seed = 42
random.seed(seed)
np.random.seed(seed)
warnings.filterwarnings('ignore')  # Suppress warning messages for cleaner output

Load Data:

In [2]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

Data Analysis:

In [3]:
train_df.sort_index()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


Replace any "?" with nan values (on the off chance they weren't already replaced with nan)

In [4]:
#Check if cell has "?", if so -> replace with nan
train_df_nan = train_df.replace("?", np.nan)
test_df_nan = test_df.replace("?", np.nan)

#Check if cell is empty, if so -> replace with nan
train_df_nan = train_df_nan.replace('', np.nan).fillna('nan')
test_df_nan = test_df_nan.replace('', np.nan).fillna('nan')

#Replace %20 with a space
train_df_nan = train_df_nan.replace('%20', ' ', regex=True)
test_df_nan = test_df_nan.replace('%20', ' ', regex=True)

#Replace '304' (presumably an error value, found in location column) with nan
train_df_nan = train_df_nan.replace('304', np.nan).fillna('nan')
test_df_nan = test_df_nan.replace('304', np.nan).fillna('nan')



train_df_nan.shape

(7613, 5)

In [5]:
def clean_text(text):
    text = str (text).lower()
    text = re.sub(r'https?://\S+|www\.\S+','',text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

In [9]:
train_df_nan['clean_text'] = train_df_nan['text'].apply(clean_text)
test_df_nan['clean_text'] = test_df_nan['text'].apply(clean_text)
test_df_nan['clean_location'] = train_df_nan['location'].apply(clean_text)
test_df_nan['clean_location'] = test_df_nan['location'].apply(clean_text)
test_df_nan['clean_Keyword'] = test_df_nan['keyword'].apply(clean_text)
test_df_nan['clean_Keyword'] = test_df_nan['keyword'].apply(clean_text)

In [10]:
train_df_nan.info()
train_df_nan.describe()
train_df_nan.to_csv("example.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          7613 non-null   int64 
 1   keyword     7613 non-null   object
 2   location    7613 non-null   object
 3   text        7613 non-null   object
 4   target      7613 non-null   int64 
 5   clean_text  7613 non-null   object
dtypes: int64(2), object(4)
memory usage: 357.0+ KB


In [8]:
train_df_nan.sort_index()

,id,keyword,location,text,target,clean_text
0,1,nan,nan,Our Deeds are the Reason of this #earthquake M...,1,our deeds are the reason of this earthquake ma...
1,4,nan,nan,Forest fire near La Ronge Sask. Canada,1,forest fire near la ronge sask canada
2,5,nan,nan,All residents asked to 'shelter in place' are ...,1,all residents asked to shelter in place are be...
3,6,nan,nan,"13,000 people receive #wildfires evacuation or...",1,13000 people receive wildfires evacuation orde...
4,7,nan,nan,Just got sent this photo from Ruby #Alaska as ...,1,just got sent this photo from ruby alaska as s...
...,...,...,...,...,...,...
7608,10869,nan,nan,Two giant cranes holding a bridge collapse int...,1,two giant cranes holding a bridge collapse int...
7609,10870,nan,nan,@aria_ahrary @TheTawniest The out of control w...,1,ariaahrary thetawniest the out of control wild...
7610,10871,nan,nan,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1,m194 0104 utc5km s of volcano hawaii
7611,10872,nan,nan,Police investigating after an e-bike collided ...,1,police investigating after an ebike collided w...


Categorize the (very few) features:

In [ ]:
numeric_features = []
categorical_features = []
ordinal_features = []
binary_features = []
drop_features = []  # do not include these features in modeling
passthrough_features = []  # do not apply any transformation

# Example: numeric_features = ["age"]
# BEGIN YOUR CODE

numeric_features = ["id"]
categorical_features = ["keyword", "location"]
ordinal_features = []
binary_features = []
drop_features = []  # do not include these features in modeling
passthrough_features = []  # do not apply any transformation

# END YOUR CODE

target = "target"

Separate Features (mostly just separating out the target feature):

In [ ]:
X_train = train_df_nan.drop(['target'], axis=1)
y_train = train_df_nan['target']
#X_test = test_df_nan.drop(['target'], axis=1)
#y_test = test_df_nan['target']

Create Classifier:

In [ ]:
dummy_majority = dummyClassifier(strategy='most_frequent')

dummy_majority.fit(x_train, y_train)

pred_majority = dummy_majority.predict(X_val)